In [2]:
from pathlib import Path
import sys

# Go from notebooks/ to project root
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

c:\MedVision-AI


"""
==============================================================
Notebook 11

Batch Explainability Validation

MedVision-AI

This notebook validates the trained EfficientNet model on
multiple unseen test images by generating

• Predictions
• Grad-CAM
• JSON Reports
• CSV Summary

==============================================================
"""

In [3]:
# ============================================================
# Standard Library Imports
# ============================================================

from pathlib import Path

# ============================================================
# Third-Party Imports
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

import torch
from PIL import Image

# ============================================================
# Project Imports
# ============================================================

from app.models.efficientnet import (
    build_efficientnet_b0,
)

from app.preprocessing.transforms import (
    test_transform,
)

from app.training.checkpoint import (
    load_checkpoint,
)

from app.explainability.gradcam import (
    GradCAM,
)

from app.explainability.visualization import (
    visualize_gradcam,
    save_visualization,
)

from app.explainability.report import (
    generate_report,
    format_report,
    save_report,
)

In [4]:
PROJECT_ROOT = Path.cwd().parent

TEST_DIR = PROJECT_ROOT / "dataset" / "chest_xray" / "test"

NORMAL_DIR = TEST_DIR / "NORMAL"

PNEUMONIA_DIR = TEST_DIR / "PNEUMONIA"

VISUALIZATION_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "batch_visualizations"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "batch_reports"
)

CSV_DIR = (
    PROJECT_ROOT
    / "outputs"
)

VISUALIZATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [21]:
# ============================================================
# Output Directories
# ============================================================

OUTPUT_DIR = PROJECT_ROOT / "outputs"

VISUALIZATION_DIR = OUTPUT_DIR / "batch_visualizations"
REPORT_DIR = OUTPUT_DIR / "batch_reports"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

VISUALIZATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [5]:
# ============================================================
# Locate Best Model Checkpoint
# ============================================================

checkpoint_candidates = [
    PROJECT_ROOT / "checkpoints" / "best_model.pth",
    PROJECT_ROOT / "outputs" / "checkpoints" / "best_model.pth",
    PROJECT_ROOT / "notebooks" / "checkpoints" / "best_model.pth",
]

CHECKPOINT_PATH = None

for path in checkpoint_candidates:
    if path.exists():
        CHECKPOINT_PATH = path
        break

if CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        "Could not locate best_model.pth."
    )

print("=" * 60)
print("Checkpoint Found")
print("=" * 60)
print(CHECKPOINT_PATH)

Checkpoint Found
c:\MedVision-AI\checkpoints\best_model.pth


In [6]:
# ============================================================
# Device
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(f"Using device: {DEVICE}")

Using device: cuda


In [7]:
CLASS_NAMES = [
    "NORMAL",
    "PNEUMONIA",
]

# Build Model

Construct the EfficientNet-B0 architecture used during training.

The model architecture must exactly match the architecture used to generate the checkpoint before loading trained weights.

In [8]:
# ============================================================
# Build EfficientNet-B0
# ============================================================

model = build_efficientnet_b0(
    num_classes=2,
    pretrained=False,
    freeze_backbone=False,
)

model.to(DEVICE)

print(model.__class__.__name__)

EfficientNet


In [9]:
# ============================================================
# Load Trained Model
# ============================================================

checkpoint = load_checkpoint(
    CHECKPOINT_PATH,
    model,
)

model.to(DEVICE)

model.eval()

print("=" * 60)
print("Checkpoint Loaded Successfully")
print("=" * 60)

print(f"Epoch            : {checkpoint['epoch']}")
print(f"Validation Loss  : {checkpoint['val_loss']:.4f}")

Checkpoint Loaded Successfully
Epoch            : 5
Validation Loss  : 0.2416


In [10]:
import random

random.seed(42)

NORMAL_DIR = (
    PROJECT_ROOT
    / "dataset"
    / "chest_xray"
    / "test"
    / "NORMAL"
)

PNEUMONIA_DIR = (
    PROJECT_ROOT
    / "dataset"
    / "chest_xray"
    / "test"
    / "PNEUMONIA"
)

normal_images = sorted(
    NORMAL_DIR.glob("*.jpeg")
)

pneumonia_images = sorted(
    PNEUMONIA_DIR.glob("*.jpeg")
)

selected_images = (
    random.sample(normal_images,5)
    +
    random.sample(pneumonia_images,5)
)

print(f"Selected {len(selected_images)} images")

Selected 10 images


In [11]:
results = []

target_layer = model.features[-1]

gradcam = GradCAM(
    model,
    target_layer,
)

In [14]:
# ============================================================
# Batch Prediction and Explainability
# ============================================================

for image_path in selected_images:

    # --------------------------------------------------------
    # True Label
    # --------------------------------------------------------
    true_label = image_path.parent.name

    # --------------------------------------------------------
    # Load Image
    # --------------------------------------------------------
    image = Image.open(image_path).convert("RGB")

    # --------------------------------------------------------
    # Preprocess Image
    # --------------------------------------------------------
    input_tensor = (
        test_transform(image)
        .unsqueeze(0)
        .to(DEVICE)
    )

    # --------------------------------------------------------
    # Forward Pass
    # (Do NOT use torch.no_grad() because Grad-CAM needs gradients)
    # --------------------------------------------------------
    outputs = model(input_tensor)

    probabilities = torch.softmax(outputs, dim=1)

    confidence, prediction = torch.max(
        probabilities,
        dim=1,
    )

    predicted_label = CLASS_NAMES[prediction.item()]
    confidence = confidence.item()

    # --------------------------------------------------------
    # Generate Grad-CAM Heatmap
    # --------------------------------------------------------
    heatmap = gradcam.generate(
        image_tensor=input_tensor,
        target_class=prediction.item(),
    )

    # --------------------------------------------------------
    # Visualize Grad-CAM
    # --------------------------------------------------------
    image_np = np.array(image)

    figure = visualize_gradcam(
        image=image_np,
        heatmap=heatmap,
    )

    # --------------------------------------------------------
    # Save Visualization
    # --------------------------------------------------------
    visualization_path = (
        VISUALIZATION_DIR
        / f"{image_path.stem}_gradcam.png"
    )

    save_visualization(
        figure=figure,
        filepath=visualization_path,
        close=True,
    )

    # --------------------------------------------------------
    # Generate Prediction Report
    # --------------------------------------------------------
    report = generate_report(
        image_name=image_path.name,
        predicted_class=predicted_label,
        confidence=confidence,
    )

    # --------------------------------------------------------
    # Save Prediction Report
    # --------------------------------------------------------
    report_path = (
        REPORT_DIR
        / f"{image_path.stem}.json"
    )

    save_report(
        report=report,
        filepath=report_path,
    )

    # --------------------------------------------------------
    # Store Results
    # --------------------------------------------------------
    results.append(
        {
            "Image": image_path.name,
            "True Label": true_label,
            "Prediction": predicted_label,
            "Confidence (%)": round(
                confidence * 100,
                2,
            ),
            "Correct": (
                true_label == predicted_label
            ),
        }
    )

    # --------------------------------------------------------
    # Print Result
    # --------------------------------------------------------
    print("-" * 60)
    print(f"Image      : {image_path.name}")
    print(f"True Label : {true_label}")
    print(f"Prediction : {predicted_label}")
    print(f"Confidence : {confidence:.2%}")
    print(f"Correct    : {true_label == predicted_label}")

------------------------------------------------------------
Image      : NORMAL2-IM-0288-0001.jpeg
True Label : NORMAL
Prediction : NORMAL
Confidence : 72.14%
Correct    : True
------------------------------------------------------------
Image      : IM-0036-0001.jpeg
True Label : NORMAL
Prediction : NORMAL
Confidence : 82.52%
Correct    : True
------------------------------------------------------------
Image      : IM-0010-0001.jpeg
True Label : NORMAL
Prediction : PNEUMONIA
Confidence : 66.10%
Correct    : False
------------------------------------------------------------
Image      : NORMAL2-IM-0326-0001.jpeg
True Label : NORMAL
Prediction : NORMAL
Confidence : 99.57%
Correct    : True
------------------------------------------------------------
Image      : NORMAL2-IM-0012-0001.jpeg
True Label : NORMAL
Prediction : NORMAL
Confidence : 99.86%
Correct    : True
------------------------------------------------------------
Image      : person141_bacteria_676.jpeg
True Label : PNEUMON

In [18]:
# ============================================================
# Cleanup
# ============================================================

gradcam.remove_hooks()

print("Grad-CAM hooks removed successfully.")

Grad-CAM hooks removed successfully.


In [19]:
# ============================================================
# Create Results DataFrame
# ============================================================

results_df = pd.DataFrame(results)

print("=" * 60)
print("Batch Results DataFrame Created")
print("=" * 60)

results_df.head()

Batch Results DataFrame Created


,Image,True Label,Prediction,Confidence (%),Correct
0,NORMAL2-IM-0288-0001.jpeg,NORMAL,NORMAL,72.14,True
1,IM-0036-0001.jpeg,NORMAL,NORMAL,82.52,True
2,IM-0010-0001.jpeg,NORMAL,PNEUMONIA,66.10,False
3,NORMAL2-IM-0326-0001.jpeg,NORMAL,NORMAL,99.57,True
4,NORMAL2-IM-0012-0001.jpeg,NORMAL,NORMAL,99.86,True


In [22]:
# ============================================================
# Save Batch Validation Results
# ============================================================

CSV_PATH = (
    OUTPUT_DIR
    / "batch_validation_results.csv"
)

results_df.to_csv(
    CSV_PATH,
    index=False,
)

print("=" * 60)
print("Results Saved Successfully")
print("=" * 60)
print(CSV_PATH)

Results Saved Successfully
c:\MedVision-AI\outputs\batch_validation_results.csv


In [23]:
# ============================================================
# Batch Validation Summary
# ============================================================

total_images = len(results_df)

correct_predictions = results_df["Correct"].sum()

accuracy = (
    correct_predictions
    / total_images
) * 100

print("=" * 60)
print("Batch Validation Summary")
print("=" * 60)

print(f"Total Images        : {total_images}")
print(f"Correct Predictions : {correct_predictions}")
print(f"Accuracy            : {accuracy:.2f}%")

print("=" * 60)

Batch Validation Summary
Total Images        : 10
Correct Predictions : 9
Accuracy            : 90.00%


In [24]:
# ============================================================
# Display Validation Results
# ============================================================

results_df

,Image,True Label,Prediction,Confidence (%),Correct
0,NORMAL2-IM-0288-0001.jpeg,NORMAL,NORMAL,72.14,True
1,IM-0036-0001.jpeg,NORMAL,NORMAL,82.52,True
2,IM-0010-0001.jpeg,NORMAL,PNEUMONIA,66.10,False
3,NORMAL2-IM-0326-0001.jpeg,NORMAL,NORMAL,99.57,True
4,NORMAL2-IM-0012-0001.jpeg,NORMAL,NORMAL,99.86,True
5,person141_bacteria_676.jpeg,PNEUMONIA,PNEUMONIA,98.28,True
6,person138_bacteria_658.jpeg,PNEUMONIA,PNEUMONIA,91.11,True
7,person123_bacteria_587.jpeg,PNEUMONIA,PNEUMONIA,99.42,True
8,person92_bacteria_451.jpeg,PNEUMONIA,PNEUMONIA,99.78,True
9,person119_bacteria_566.jpeg,PNEUMONIA,PNEUMONIA,64.45,True


In [25]:
# ============================================================
# Notebook Completed
# ============================================================

print("=" * 60)
print("MedVision-AI Batch Explainability Validation Completed")
print("=" * 60)

print(f"Images Evaluated : {total_images}")
print(f"Accuracy         : {accuracy:.2f}%")
print(f"CSV Summary      : {CSV_PATH}")
print(f"Grad-CAM Folder  : {VISUALIZATION_DIR}")
print(f"Reports Folder   : {REPORT_DIR}")

print("=" * 60)

MedVision-AI Batch Explainability Validation Completed
Images Evaluated : 10
Accuracy         : 90.00%
CSV Summary      : c:\MedVision-AI\outputs\batch_validation_results.csv
Grad-CAM Folder  : c:\MedVision-AI\outputs\batch_visualizations
Reports Folder   : c:\MedVision-AI\outputs\batch_reports
